In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

print("Libraries loaded")

Libraries loaded


In [4]:
customers = pd.read_csv("../data/customer_profiles.csv")

accounts = pd.read_csv("../data/bank_accounts.csv")

In [6]:
transactions = pd.read_csv(
    "../data/account_transactions.csv",
    nrows=100000
)

In [7]:
print("Customers:", customers.shape)
print("Accounts:", accounts.shape)
print("Transactions:", transactions.shape)

Customers: (1000, 5)
Accounts: (3553, 9)
Transactions: (100000, 8)


In [8]:
print(customers.columns.tolist())
print(accounts.columns.tolist())
print(transactions.columns.tolist())

['customer_id', 'national_id', 'birth_date', 'city', 'state']
['account_id', 'customer_id', 'account_type', 'creation_date', 'account_status', 'balance', 'loan_amount', 'term_months', 'interest_rate']
['transaction_id', 'account_id', 'linked_transaction_id', 'created_time', 'transaction_type', 'transaction_code', 'amount', 'channel']


In [9]:
customers.info()

accounts.info()

transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  1000 non-null   str  
 1   national_id  1000 non-null   str  
 2   birth_date   1000 non-null   str  
 3   city         1000 non-null   str  
 4   state        1000 non-null   str  
dtypes: str(5)
memory usage: 111.5 KB
<class 'pandas.DataFrame'>
RangeIndex: 3553 entries, 0 to 3552
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   account_id      3553 non-null   str    
 1   customer_id     3553 non-null   str    
 2   account_type    3553 non-null   str    
 3   creation_date   3553 non-null   str    
 4   account_status  3553 non-null   str    
 5   balance         3553 non-null   float64
 6   loan_amount     1007 non-null   float64
 7   term_months     2553 non-null   float64
 8   interest_rate   2553 non-null   float6

In [10]:
customers.isnull().sum()
accounts.isnull().sum()
transactions.isnull().sum()

transaction_id               0
account_id                   0
linked_transaction_id    98492
created_time                 0
transaction_type             0
transaction_code             0
amount                       0
channel                      0
dtype: int64

In [11]:
transactions = pd.read_csv("../data/account_transactions.csv")
codes = pd.read_csv("../data/transaction_codes.csv")

In [12]:
df = transactions.merge(codes, on="transaction_code", how="left")

In [13]:
df.head()

,transaction_id,account_id,linked_transaction_id,created_time,transaction_type_x,transaction_code,amount,channel_x,label,account_type,transaction_type_y,channel_y
0,f5526a0f-4ec2-4aa4-a01e-33034d40ec33,eb1f2f49-1bc0-45c8-8991-4abea8f7f6ea,NaN,2023-04-21 12:55:00,Credit,IBK_IN,5750.20,InterbankPortal,Interbank Transfer In,Savings,Credit,InterbankPortal
1,b1c5757f-587d-418b-9238-35c4e338b731,eb1f2f49-1bc0-45c8-8991-4abea8f7f6ea,NaN,2023-04-22 11:18:00,Credit,IBK_IN,3253.85,InterbankPortal,Interbank Transfer In,Savings,Credit,InterbankPortal
2,f248fe0e-b646-4df6-a9d8-e8ebcd386c92,eb1f2f49-1bc0-45c8-8991-4abea8f7f6ea,NaN,2023-04-22 12:16:00,Credit,IBK_IN,1055.74,InterbankPortal,Interbank Transfer In,Savings,Credit,InterbankPortal
3,44314720-e02d-4c39-8aaf-b0d81380225d,eb1f2f49-1bc0-45c8-8991-4abea8f7f6ea,NaN,2023-04-22 14:15:00,Debit,CSH_WDL,400.00,Branch,Cash Withdrawal,Savings,Debit,"ATM,Branch"
4,9dcabefa-fe70-4d3f-b29e-22c442cba3dd,eb1f2f49-1bc0-45c8-8991-4abea8f7f6ea,NaN,2023-04-22 16:24:00,Debit,CSH_WDL,480.00,Branch,Cash Withdrawal,Savings,Debit,"ATM,Branch"


In [15]:
df["transaction_type"] = df["transaction_type_x"].fillna(df["transaction_type_y"])
df["channel"] = df["channel_x"].fillna(df["channel_y"])

In [16]:
df.drop(columns=[
    "transaction_type_x",
    "transaction_type_y",
    "channel_x",
    "channel_y"
], inplace=True)

In [17]:
df.amount.sum()

np.float64(2165793344.49)

In [18]:
df["transaction_type"].value_counts(normalize=True)

transaction_type
Debit     0.743026
Credit    0.256974
Name: proportion, dtype: float64

In [19]:
df["channel"].value_counts

<bound method IndexOpsMixin.value_counts of 0          InterbankPortal
1          InterbankPortal
2          InterbankPortal
3                   Branch
4                   Branch
                ...       
2377164    InterbankPortal
2377165      OnlineBanking
2377166             Branch
2377167             Branch
2377168             System
Name: channel, Length: 2377169, dtype: str>

In [21]:
df.groupby("transaction_type")["amount"].sum()

transaction_type
Credit    1.244265e+09
Debit     9.215280e+08
Name: amount, dtype: float64

In [22]:
df.groupby("label")["amount"].sum().sort_values(ascending=False)

label
Interbank Transfer In       6.507923e+08
Interbank Transfer Out      4.683794e+08
Cash Deposit                3.234108e+08
Cash Withdrawal             1.568357e+08
Merchant Payment            1.281728e+08
Loan Disbursement           7.676845e+07
Loan Disbursement Credit    7.676845e+07
FD Opening Credit           7.465320e+07
FD Opening Debit            4.275863e+07
FD Maturity Credit          4.187206e+07
FD Maturity Liquidation     4.056106e+07
Loan Repayment Debit        3.769347e+07
Loan Repayment              3.769347e+07
Loan Interest Charge        9.391416e+06
Annual Maintenance Fee      4.201637e+04
Name: amount, dtype: float64

In [23]:
df["created_time"] = pd.to_datetime(df["created_time"])

In [24]:
df["hour"] = df["created_time"].dt.hour
df["day"] = df["created_time"].dt.day_name()
df["month"] = df["created_time"].dt.month

In [25]:
df.groupby("hour")["amount"].count()

hour
7      31682
8      95124
9     158296
10    222010
11    285336
12    301736
13    268990
14    238105
15    205624
16    174425
17    142505
18    110984
19     78887
20     47703
21     15762
Name: amount, dtype: int64

In [26]:
df.groupby("hour")["amount"].mean()

hour
7      932.673013
8      908.946259
9      891.434601
10     930.573151
11     894.109110
12     920.730926
13     902.420801
14     921.836538
15     916.316974
16     904.646241
17     892.932354
18     916.785627
19     913.255221
20     917.143141
21    1008.722634
Name: amount, dtype: float64

In [27]:
df["channel"].value_counts()

channel
OnlineBanking      491988
MobileApp          491964
InterbankPortal    476037
Branch             465789
ATM                430995
System              20396
Name: count, dtype: int64

In [28]:
df.groupby("channel")["amount"].sum().sort_values(ascending=False)

channel
InterbankPortal    1.119172e+09
Branch             5.839869e+08
ATM                2.425953e+08
System             9.186656e+07
OnlineBanking      6.410271e+07
MobileApp          6.407013e+07
Name: amount, dtype: float64